# Autoresearch MLflow Stock Model

- Dataset: `data/runs/phase2-source-20260424`
- Evaluator: `scripts/autoresearch_eval.py`
- MLflow tracking URI: `sqlite:///data/mlflow/mlflow.db`
- Primary metric: `sharpe_diff`
- Secondary metrics: `information_ratio`, `active_return`, `sharpe_diff_ci_low`
- Validation: fixed walk-forward portfolio backtest with SPY date alignment

The generic shuffled 5-fold CV template is intentionally not used for the final comparison here because it would break the portfolio time-ordering and SPY-relative evaluation contract. The repo's walk-forward backtest is the validation authority for this experiment.


In [ ]:
from pathlib import Path

import pandas as pd

RESULTS_TSV = Path("experiments/autoresearch/results.tsv")
results = pd.read_csv(RESULTS_TSV, sep="\t")
results.tail()


In [ ]:
experiment_rows = results.loc[
    results["iteration_id"].astype(str).str.startswith("exp_"),
    [
        "candidate_id",
        "candidate_sharpe",
        "spy_sharpe",
        "sharpe_diff",
        "active_return",
        "information_ratio",
        "sharpe_diff_ci_low",
        "mean_turnover",
        "status",
    ],
].copy()
leaderboard = experiment_rows.sort_values(
    ["sharpe_diff", "information_ratio"], ascending=False
).reset_index(drop=True)
leaderboard.head(10)


## Leaderboard Snapshot

| Candidate | Sharpe | SPY Sharpe | Sharpe Diff | Active Return | IR | CI Low | Turnover | Status |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| `e8_scale_0p85` | 3.026434 | 1.130685 | 1.895749 | 0.603803 | 2.278831 | -0.399057 | 0.691957 | provisional |
| `e8_scale_0p9` | 3.023593 | 1.130685 | 1.892907 | 0.616570 | 2.263636 | -0.418502 | 0.689014 | provisional |
| `e8_scale_0p8` | 2.999766 | 1.130685 | 1.869081 | 0.586625 | 2.278141 | -0.397268 | 0.695324 | provisional |
| `e8_scale_4p0` | 2.991126 | 1.130685 | 1.860440 | 0.843023 | 2.105149 | -0.421310 | 0.671771 | provisional |
| `e8_baseline` | 2.987197 | 1.130685 | 1.856511 | 0.636036 | 2.212435 | -0.445541 | 0.686668 | provisional |


## Winner

Winner model: `e8_scale_0p85`

Family: Ridge plus LightGBM E8 blend with score calibration

Primary metric: `sharpe_diff = 1.895749`, up from baseline `1.856511`

Secondary metrics: `information_ratio = 2.278831`, up from baseline `2.212435`; `active_return = 0.603803`, still positive but lower than baseline `0.636036`; `sharpe_diff_ci_low = -0.399057`, still provisional.

Validation detail: 46 aligned SPY observations from the fixed walk-forward evaluator, top 100 assets, 48 requested rebalances, 5 bps transaction costs, optimizer max weight 0.30.

Why it won: scaling E8 forecast scores by 0.85 improved risk-adjusted allocation enough to lift Sharpe and SPY-relative IR without changing labels, benchmark alignment, or evaluator logic.

Next iteration: run broader confirmation coverage and then promote the same score scaling into `src/stock_analysis/forecasting/ml_forecast.py` only if the confirmation run preserves the objective improvement.
